In [ ]:
# !pip install folium osmnx geopandas pandas shapely

import folium
import osmnx as ox
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
from datetime import datetime

In [ ]:
### CONSTS ###
TRAFFIC_DIR = "../data/traffic"

LAT = 48.14553552490184
LNG = 17.114780288953938
RADIUS = 3000  # meters

MAP_ZOOM = 14
MAP_WIDTH = 2304
MAP_HEIGHT = 2304

START_TIME = datetime(2025, 1, 31, 6, 0)
END_TIME = datetime(2025, 1, 31, 9, 30)

# Load data into traffic graph

In [ ]:
from city_graph import build_road_graph, visualize_graph

G_major, nodes_major, edges_major = build_road_graph("Bratislava, Slovakia")

m = visualize_graph(nodes_major, edges_major)
m

In [ ]:
from traffic_data import build_traffic_raster, visualize_traffic_raster

aggregated_image = build_traffic_raster(
    folder=TRAFFIC_DIR, start_time=START_TIME, end_time=END_TIME, step=4
)

traffic_map = visualize_traffic_raster(
    raster=aggregated_image,
    center_lat=48.14553552490184,
    center_lng=17.114780288953938,
    zoom=14,
    width=2304,
    height=2304,
)
traffic_map

In [ ]:
from traffic_graph import add_congestion_to_graph, visualize_weighted_graph


G_weighted = add_congestion_to_graph(
    graph=G_major,
    raster_image=aggregated_image,
    center_lat=LAT,
    center_lng=LNG,
    zoom=MAP_ZOOM,
)
print(G_weighted)

map_congestion = visualize_weighted_graph(G_weighted, center_lat=LAT, center_lng=LNG)
map_congestion

In [ ]:
from vehicle_data import load_vehicle_positions, vehicle_heatmap

vehicles = load_vehicle_positions("../data/vehicles/merged_vehicles_piatok.csv", START_TIME, END_TIME)
vehicles.head()
         
m = vehicle_heatmap(vehicles, center_lat=LAT, center_lng=LNG)
m

In [ ]:
from filter import crop_graph_radius

G_center = crop_graph_radius(
    G_weighted, center_lat=LAT, center_lng=LNG, radius_m=RADIUS
)

map = visualize_weighted_graph(G_center, center_lat=48.1486, center_lng=17.1077)
map

# Deployments

In [ ]:
from rsu.static_deployment import deploy_static_rsus
from rsu.visualize import visualize_rsu_deployment

NUM_RSUS = 10
COVERAGE_RADIUS = 300  # meters (urban DSRC approx)

rsu_nodes_static = deploy_static_rsus(
    graph=G_center, num_rsus=NUM_RSUS, coverage_radius_m=COVERAGE_RADIUS
)
print(rsu_nodes_static)

m = visualize_rsu_deployment(
    graph=G_center,
    rsu_nodes=rsu_nodes_static,
    center_lat=LAT,
    center_lng=LNG
)
m

In [ ]:
from rsu.cluster_deployment import deploy_cluster_head_rsus

rsu_nodes_cluster = deploy_cluster_head_rsus(G_center, num_rsus=10)

m = visualize_rsu_deployment(
    graph=G_center, rsu_nodes=rsu_nodes_cluster, center_lat=LAT, center_lng=LNG
)
m

In [ ]:
from rsu.hybrid_deployment import hybrid_rsu_deployment
from rsu.visualize import visualize_hybrid_deployment

S_static, M_mobile = hybrid_rsu_deployment(
    G_center, vehicles, total_rsu_budget=75, srsu_radius=300, mrsu_radius=500
)

rsu_nodes_hybrid = S_static + M_mobile

m = visualize_hybrid_deployment(
    G=G_center, rsu_nodes=M_mobile, vehicles_df=vehicles, center_lat=LAT, center_lng=LNG, vehicle_radius=300
)
m

# Evaluation

In [ ]:
from evaluation import build_comparison_table, plot_coverage, run_experiment
from rsu.hybrid_deployment import hybrid_rsu_deployment
from rsu.cluster_deployment import deploy_cluster_head_rsus
from rsu.static_deployment import deploy_static_rsus

strategies = {
    "static": deploy_static_rsus,
    "cluster": deploy_cluster_head_rsus,
    "hybrid": hybrid_rsu_deployment,
}

budgets = range(5, 101, 5)

df = df = run_experiment(
    G_center, vehicles, strategies, budgets, r_static=300, r_mobile=450
)

# Figures

In [ ]:
from figures import _save_map_png

m = visualize_weighted_graph(G_weighted, center_lat=LAT, center_lng=LNG)
folium.Circle(
    location=[LAT, LNG], radius=RADIUS*10, color="yellow", fill=False, weight=2
).add_to(m)
_save_map_png(m, "img/figure_traffic_graph.png")

m = visualize_hybrid_deployment(
    G=G_weighted,
    rsu_nodes=[],
    vehicles_df=vehicles,
    center_lat=LAT,
    center_lng=LNG,
    vehicle_radius=300,
)
folium.Circle(
    location=[LAT, LNG], radius=RADIUS * 10, color="yellow", fill=False, weight=2
).add_to(m)
_save_map_png(m, "img/figure_vehicles_graph.png")

In [ ]:
from evaluation import plot_combined_coverage, plot_coverage_curves_seaborn, plot_graph_coverage_seaborn, plot_guaranteed_ratio, plot_hybrid_allocation
import pandas as pd

df = pd.read_csv("rsu_experiment_results.csv")

plot_coverage_curves_seaborn(df)
plot_graph_coverage_seaborn(df)
plot_hybrid_allocation(df)
plot_guaranteed_ratio(df) 

plot_combined_coverage(df, save_path="img/coverage_curves.png")